In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.prompts import PromptTemplate
from dotenv import dotenv_values

config = dotenv_values(".env")
llm = ChatOpenAI(
    api_key=config["OPENROUTER_API"],
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free",
    temperature=0.8,
)

In [3]:
# loading
from langchain_classic.document_loaders import TextLoader
loader = TextLoader("osama_eslam_personal_knowledge_base.txt")
document = loader.load()

# chunking
from langchain_classic.text_splitter import TokenTextSplitter
splitter = TokenTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(document)


# embedding
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
embedding = HuggingFaceBgeEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device":"cpu"}
)

vectors = embedding.embed_documents([i.page_content for i in chunks])

# vector store
from langchain_classic.vectorstores import FAISS
vectorDB = FAISS.from_documents(chunks, embedding)

C:\Users\osama\AppData\Local\Temp\ipykernel_18256\486756336.py:18: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceBgeEmbeddings(


In [4]:
temp = """
you are assistant, asnwer the question based on the context below,
if you don't know the answer,
just say "I don't know", don't try to make up an answer.
answer in 100 words or less.

Context: {context}
question: {question}

"""
temp = PromptTemplate.from_template(temp)

query = "what is the interests of osama eslam?"

similar_docs = vectorDB.similarity_search_with_score(query, k=3)

context = []
for i in similar_docs:
    context.append(i[0].page_content)

prompt = temp.format(context="\n".join(context), question=query)
print(prompt)


you are assistant, asnwer the question based on the context below,
if you don't know the answer,
just say "I don't know", don't try to make up an answer.
answer in 100 words or less.

Context:  & Orange Digital Center (July 2026).

Q: What repositories has Osama published or contributed to on GitHub?
A: Key repositories include:
- DoCrops Graduation Project: https://github.com/Osama2004o/DoCrops-FCAI-Graduation-Project
- AI Video Generation Platform: https://github.com/MoHatemTC/ai-video-generation-app
- Product Market Search Multi-Agent: https://github.com/Osama2004o/Product-Market-Search
- Mini-RAG Implementation: https://github.com/Osama2004o/mini-rag
- Personal GitHub Profile: https://github.com/Osama2004o

Q: What internships and professional experiences has Osama completed?
A: Osama's experience includes:
- SpaceTech (Sep 2026 â€“ Present): AI Engineer Intern (Remote)
- Sprints.ai (Jul 2026 â€“ Aug 2026): AI Engineering Intern (Remote)
- Tips Hindawi (Dec 2025 â€“ Feb 2026): LLM

In [5]:
response = llm.invoke(prompt).content

In [6]:
from IPython.display import display, Markdown
display(Markdown(response))




Osama's professional interests and core focus areas include Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), Multi-Agent Systems, Computer Vision, Edge ML Inference, and Backend Systems Engineering (Python, FastAPI, ASP.NET Core). He specializes in designing, evaluating, and deploying end-to-end LLM applications, production RAG pipelines, and multi-agent workflows. His work also centers on Computer Vision and Edge ML Inference, such as the offline DoCrops crop disease classifier.